# 🦉 Notebook 1: Stack Overflow — Class Design

**Stack Overflow** is a Q&A site where users ask questions, post answers, vote, earn **reputation**,
and unlock privileges with **badges**. It's a popular **object-oriented design (OOD)** interview question.

In a real interview, the grade comes *less* from "did your code run" and *more* from how you:

1. Clarify the requirements (ask questions!).
2. Identify **actors** and **use-cases**.
3. Pick out **entities** (nouns) and **behaviors** (verbs).
4. Draw how classes relate.
5. Sanity-check against **SOLID** principles.

This notebook walks those steps. Notebook 2 turns the design into runnable code
(bad → good → best). Notebook 3 adds real-world extensions.


## 🛠️ Setup

```bash
cd 07-object-oriented-design/stack-overflow
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1. Clarifying questions

Always start by asking. A few good ones for Stack Overflow:

- **Users**: are there *guests* (read-only) vs *members* (post/vote) vs *moderators* (close/reopen) vs *admins* (ban)?
- **Posts**: can both questions and answers receive **comments**? How long are they?
- **Voting**: can a user **change** their vote? **Remove** it? Vote on their own posts (usually *no*)?
- **Reputation**: how many points per up-vote on a question vs an answer? Down-vote penalty?
- **Accepted answers**: can the asker accept multiple answers, or only one?
- **Tags**: free-form or curated? Is there a limit per question?
- **Badges**: automatic (based on reputation milestones) or manually awarded?
- **Search**: by text? by tag? sorted how? (active / newest / votes)
- **Moderation**: who can close/reopen/delete questions? What are the reasons?
- **Scale**: is this a single-process exercise, or do we need to think about DBs and sharding? *(For an OOD interview: start single-process.)*

> ⚖️ **Rule of thumb:** it's better to ask three questions and design the right thing,
> than to race ahead and design for assumptions the interviewer didn't make.


## 2. Actors and use-cases

An **actor** is anyone (or anything) that interacts with the system.

| Actor      | What they do |
|------------|--------------|
| Guest      | Search and read questions. Can't post or vote. |
| Member     | Everything a guest can do, plus: post questions/answers, add comments, vote, accept answers on their own questions. |
| Moderator  | Everything a member can do, plus: close/reopen/delete any question. |
| Admin      | Block or unblock members. |
| System     | Awards badges, sends notifications. |

### Core use-cases (happy path)

1. `ask_question(title, body, tags)` → returns a `Question`.
2. `post_answer(question, body)` → returns an `Answer`.
3. `vote(post, UP | DOWN)` → updates the post's score and the author's reputation.
4. `accept_answer(question, answer)` → only the asker; only one accepted answer.
5. `comment(post, text)` → short free-form note attached to a Question or Answer.
6. `search(text=..., tag=...)` → list of questions.
7. `close_question(question, reason)` → moderator action.


## 3. Entities (the "nouns")

Nouns from the requirements become classes:

- **User** — `id`, `name`, `reputation`, `badges`.
- **Post** — abstract base class for things that can be voted on and commented on. Has `body`, `author`, `votes`, `comments`.
- **Question** *(a Post)* — adds `title`, `tags`, `answers`, `accepted_answer`, `status` (open / closed / deleted).
- **Answer** *(a Post)* — adds a back-reference to its `Question`.
- **Comment** — short text attached to any Post.
- **Vote** — represented as a `user_id -> UP/DOWN` map on each Post (simpler than a full class for an intro lab).
- **Tag** — for this lab, just a string. A real system would make `Tag` its own entity with description, follower-count, etc.
- **Badge** — name + rule (e.g., *"Nice Answer" = any answer reaches score 10*).

### "Is-a" vs "has-a"

- `Question` **is-a** `Post` ➜ inheritance.
- `Answer` **is-a** `Post` ➜ inheritance.
- `Question` **has-a** list of `Answer` ➜ composition.
- `Question` and `Answer` **have-many** `Comment` ➜ composition.
- `User` **has-many** `Badge`s ➜ composition.


## 4. UML-ish class diagram

```
        User (1) ---authors---> (*) Post  [abstract]
                                    |
                                    +-- has many --> Comment
                                    |
                          +---------+---------+
                          |                   |
                      Question  ---(*)---> Answer
                      title, tags, status, accepted_answer
```

Key relationships:

- `User` 1..* `Post`   (a user authors many posts)
- `Post` has-many `Comment`
- `Question` 1..* `Answer`   (a question has many answers)
- `Question` 0..1 `Answer`   (the accepted answer, if any)

> 🧠 **Why the shared `Post` base class?**
> Voting, commenting, and scoring are identical for questions and answers.
> Putting that logic once on `Post` avoids duplication and respects the **DRY** principle.


## 5. Sanity-check: SOLID

| Principle | How our design honors it |
|-----------|--------------------------|
| **S**ingle responsibility | `User` tracks identity & reputation; `Post` tracks votes/comments; `Question` orchestrates answers; a `ReputationPolicy` (notebook 3) owns scoring rules. |
| **O**pen/closed | Adding a new post-like thing (e.g., `WikiPost`) doesn't require modifying `User` or `Question` — we just subclass `Post`. |
| **L**iskov substitution | Anywhere that expects a `Post`, an `Answer` or a `Question` works (same `score`, same `comment()`). |
| **I**nterface segregation | Guests don't need `vote()`; our design keeps voting as a *function*, not a method guests have to expose. |
| **D**ependency inversion | In notebook 3 we'll inject a **notifier** and a **reputation policy**, so `Question` depends on *abstractions*, not hard-coded rules. |


## 6. What we'll build next

- **Notebook 2** — turn this design into real Python.
  We'll start with a **bad** "god class" (everything mashed into one object), then refactor
  to a **good** version (proper classes), and finish with a **best** version (enums, statuses,
  guarded self-votes, etc.).
- **Notebook 3** — real-world extensions:
  - Tags + search
  - Badges via the **observer pattern**
  - Closing / reopening questions (moderation)
  - Thread-safe voting
  - Pluggable reputation rules (strategy pattern)

> 💡 This notebook has no code on purpose — design first, code second.
